# Stat 220, Unit 2 Homework: A Map of Models

`campus_cafe.csv` is 700 days at a campus coffee shop: `temp_f`, `exam_week`,
`promo`, `foot_traffic`, `drinks_sold`, `revenue`, and `sold_out`.

```python
import pandas as pd, numpy as np
import statsmodels.api as sm, statsmodels.formula.api as smf
from sklearn.ensemble import RandomForestRegressor
from sklearn.model_selection import KFold, cross_val_score
from sklearn.linear_model import LinearRegression

cafe = pd.read_csv("https://drbob-richardson.github.io/stat220/F2026/data/campus_cafe.csv")
cafe.head()
```

**Problem 1.** *Reading a situation.* No computer.

Part a. The owner wants to know how many drinks to prepare tomorrow, given the forecast temperature. What type is `y` here, and which model family would you name?

_Your answer:_
The type of Y is count or rate, and the model family is Poisson


Part b. The owner wants to know whether running a promotion actually raises revenue, since she is deciding whether to keep doing it. What type is `y`, which family, and what makes this a different job from part a?

_Your answer:_
Since you are measuring revenue which is dollars it is the type of Y: Numbers. And the model family is Linear regression. The difference is part a is a prediction job where part b is an inference job.

Part c. The owner wants the chance they run out of an item on a given day. What type is `y`, and which family?

_Your answer:_
Type of Y is "Yes or no" and the probabilty model is logistic regression.


Part d. A classmate says to just use a random forest for all three. A forest would run on all three without complaining. Say which of the three you would refuse to use it for, and why.

_Your answer:_
I would not use it for part B, becuase it works well for the prediction tasks of part a and b, but part B is asking do promos actually have an effect and because of that a random forest tree would not work as well because it asking for just one number how much does revenue increase  by, and the sides say the job, not the outcome, rules out the model.


**Problem 2.** *What happens if you ignore the type of `y`.*

Part a. `sold_out` is a yes/no column. Describe what the linear model produced that the logistic model did not, and say why that is a problem.

In [3]:
import pandas as pd, numpy as np
import statsmodels.api as sm, statsmodels.formula.api as smf
from sklearn.ensemble import RandomForestRegressor
from sklearn.model_selection import KFold, cross_val_score
from sklearn.linear_model import LinearRegression

cafe = pd.read_csv("https://drbob-richardson.github.io/stat220/F2026/data/campus_cafe.csv")
cafe.head()

#the right model for a yes/no outcome
p_ok = smf.logit("sold_out ~ drinks_sold", data=cafe).fit(disp=0).predict()

# the same outcome, forced into a linear model
p_bad = smf.ols("sold_out ~ drinks_sold", data=cafe).fit().predict()

print(f"logistic gives values from {p_ok.min():.3f} to {p_ok.max():.3f}")
print(f"linear   gives values from {p_bad.min():.2f} to {p_bad.max():.2f}")
print("linear values that are not possible probabilities:",
      int(((p_bad < 0) | (p_bad > 1)).sum()), "out of", len(p_bad))

logistic gives values from 0.026 to 0.971
linear   gives values from -0.19 to 1.15
linear values that are not possible probabilities: 25 out of 700


_Your answer:_
The linear model produced predictions below 0, -0.19, and above 1, 1.15,resulting in 25 out of 700 invalid predictions. The logistic model produced predictions bounded between 0.026 and 0.971. It doesn't make any sense because a percentage cannot be less then 0 or over 100% which makes the results unusable.


Part b. `drinks_sold` is a count. Name the probability model you would use for it, and say in one sentence what would go wrong with an ordinary linear model there.

_Your answer:_
I would use a Poisson regression model and a ordinary linear model would go wrong is it uses an unbounded line that can go into the negatives.


**Problem 3.** *What a model can and cannot hand back.*

Part a. List the quantities the forest could not produce, and explain why not. Your answer should use the word distribution.

In [4]:
X = cafe[["drinks_sold", "exam_week", "promo", "temp_f"]]
reg = LinearRegression().fit(X, cafe["revenue"])
forest = RandomForestRegressor(n_estimators=300, random_state=0).fit(X, cafe["revenue"])
sm_reg = smf.ols("revenue ~ drinks_sold + exam_week + promo + temp_f", data=cafe).fit()

for name, model in [("regression", sm_reg), ("forest", forest)]:
    have = [a for a in ["pvalues", "conf_int", "aic"] if hasattr(model, a)]
    print(f"{name:<12} can give you: {have if have else 'none of them'}")

regression   can give you: ['pvalues', 'conf_int', 'aic']
forest       can give you: none of them


_Your answer:_
The random forest cannot produce p-values, confidence intervals, or AIC.

A random forest does not assume any underlying probability distribution. It just averages the answers. Without an assumed distribution, there is no way to caluclate p-values, confidence intervals, or AIC.



Part b. Go back to the three jobs in Problem 1. Which one of them could a forest not do at all, given what you just saw?

_Your answer:_
It could no complete part b of problem 1, because in order to do these we need a p-value and a confidence interval.


**Problem 4.** *Is model A better than model B?*

Part a. These three models are compared by AIC. Say which one you would keep, and what it means that one of the additions made AIC go up.

In [5]:
for f in ["revenue ~ drinks_sold",
          "revenue ~ drinks_sold + exam_week",
          "revenue ~ drinks_sold + exam_week + promo"]:
    print(f"{f:<48} AIC = {smf.ols(f, data=cafe).fit().aic:8.1f}")

revenue ~ drinks_sold                            AIC =   5492.8
revenue ~ drinks_sold + exam_week                AIC =   5442.6
revenue ~ drinks_sold + exam_week + promo        AIC =   5444.3


_Your answer:_
I would keep the "revenue ~ drinks_sold + exam_week" because promo adds extra noise without giving anything more useful more than drinks_sold and exam_week.

Part b. Say which model you would ship, and whether the result surprises you.

In [6]:
folds = KFold(5, shuffle=True, random_state=0)   # one fold object, used for both
for name, mod in [("regression", LinearRegression()),
                  ("forest", RandomForestRegressor(n_estimators=300, random_state=0))]:
    mse = -cross_val_score(mod, X, cafe["revenue"], cv=folds,
                           scoring="neg_mean_squared_error").mean()
    print(f"  {name:<12} cross-validated MSE {mse:7.1f}   RMSE {np.sqrt(mse):5.1f} dollars")

  regression   cross-validated MSE   140.2   RMSE  11.8 dollars
  forest       cross-validated MSE   171.9   RMSE  13.1 dollars


_Your answer:_

I would ship the regression model, this is a little surprising becuase I would think a more complex model might do better then a basic model like regression.

Part c. You could compare the three models in part a with AIC, but you could not use AIC for the comparison in part b. Explain why in two sentences.

_Your answer:_

Linear regression has a AIC becuase it assumes a bell-curve distribution. In a random forest AIC doesn't exist becuase it doesn't use the bell-curve distribution so it cannot be compared.

**Problem 5.** *Putting it together.* No computer.

Part a. The owner reads that a forest predicted revenue almost as well as the regression, and asks whether she should use the forest to decide about promotions. Answer her in three or four sentences.

_Your answer:_

No, she should not use the forest to make that decision. While the forest is good at making predictions, it will not give her an exact number to tell her whether she should run promotions or not. Linear regression gives her an exact number and a margin of error, and a p-value showing whether the promotion actually causes revenue to increase.
